# Fine-tuning an LLM on open access joural articles

This notebook demonstrates the full workflow for domain-specific LLM fine-tuning, classical NLP baselines, and evaluation.

In this example, we are fine-tuning GPT-2 on open-access articles from the Journal of Chemical Information and Modeling.

In [1]:
# Install requirements (uncomment if running in Colab)
# !pip install -r requirements.txt

## 1. Data Collection & Preprocessing
Fetch and prepare articles from Europe PMC.

In [2]:
# Fetch data using scripts/fetch.py
!python scripts/fetch.py --page_size 100 --output_dir ./data/europepmc

# Prepare data using scripts/prepare_text.py
!python scripts/prepare_text.py --data_dir ./data/europepmc --output_jsonl ./data/europepmc_prepared.jsonl

Found 100 PMCIDs. Saving to output directory.
Saved 100 articles to ./data/europepmc in 114.81 seconds.
Processing files: 100%|███████████████████████| 100/100 [00:01<00:00, 64.54it/s]
Processed 100 files in 1.56 seconds. Output: ./data/europepmc_prepared.jsonl


## 2. Classical NLP Baselines
Run SVM, Logistic Regression, and BERT baselines for comparison.

In [3]:
# SVM & Logistic Regression
!python scripts/examples/classical.py --prepared_jsonl ./data/europepmc_prepared.jsonl

SVM Results:
              precision    recall  f1-score   support

           0       0.50      0.25      0.33         8
           1       0.62      0.83      0.71        12

    accuracy                           0.60        20
   macro avg       0.56      0.54      0.52        20
weighted avg       0.57      0.60      0.56        20

Logistic Regression Results:
              precision    recall  f1-score   support

           0       0.50      0.62      0.56         8
           1       0.70      0.58      0.64        12

    accuracy                           0.60        20
   macro avg       0.60      0.60      0.60        20
weighted avg       0.62      0.60      0.60        20



In [4]:
# BERT Baseline
!python scripts/examples/bert.py --prepared_jsonl ./data/europepmc_prepared.jsonl

Loading weights: 100%|██████████████████████| 199/199 [00:00<00:00, 8189.67it/s]
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint.

## 3. LLM Fine-Tuning
Fine-tune GPT-2 on the prepared articles.

In [5]:
# Fine-tune GPT-2 using scripts/finetune.py
!python scripts/finetune.py --data_path ./data/europepmc_prepared.jsonl --output_dir ./models/gpt2-finetuned-europepmc --num_train_epochs 2 --temperature 1.0 --top_p 1.0 --max_seq_length 1024

Loading dataset...
Generating train split: 100 examples [00:00, 1330.59 examples/s]
Loaded dataset with 100 examples in 0.91 seconds.
Loading tokenizer and base model...
Loading weights: 100%|██████████████████████| 148/148 [00:00<00:00, 8663.50it/s]
Loaded model and tokenizer in 1.10 seconds.
Preparing batches from pre-tokenized 'tokens' field...
Starting training...
  0%|                                                   | 0/100 [00:00<?, ?it/s]/opt/anaconda3/envs/torch/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.
{'loss': '3.413', 'grad_norm': '5.788', 'learning_rate': '5e-07', 'epoch': '2'} 
Writing model shards: 100%|███████████████████████| 1/1 [00:00<00:00,  3.50it/s]
{'train_runtime': '70.1', 'train_samples_per_se